# SmartPantry AI — Role 1 Research: Shelf-Life Data Sourcing

**Hnin Aung · Role 1 (Research) · ITAI 2277 · Fall 2026**

## Purpose

SmartPantry AI needs to know how long each food lasts in the refrigerator. Role 3 built the
first version of that data from the **FoodSafety.gov Cold Food Storage Chart**.

While sourcing produce and dairy numbers for the Phase 3 demo, I found that chart does not
cover those food groups. This notebook documents where the numbers actually came from, so that
every shelf-life value in our app can be traced back to a government source.

## Why this part of the app is deliberately rule-based

Shelf-life values are looked up, not predicted. A food safety answer has to be traceable:
if someone asks why the app says 7 days, we can name the agency, the dataset and the date we
pulled it. A trained model cannot provide that, so we did not use one here.

## 1. The problem with our original source

The FoodSafety.gov Cold Food Storage Chart covers:

- meat, poultry, seafood
- eggs
- deli and packaged items
- leftovers

It does **not** list milk, yogurt, cheese, or any fresh fruit or vegetable. The FDA's
*Refrigerator & Freezer Storage Chart* has the same gap. Both charts point readers to a
different USDA tool for everything else: **FoodKeeper**.

FoodKeeper is published by USDA Food Safety and Inspection Service, developed with Cornell
University and the Food Marketing Institute. It covers over 650 foods, and the dataset is
public.

| | |
|---|---|
| Publisher | USDA Food Safety and Inspection Service |
| Dataset | FSIS FoodKeeper Data |
| Catalog page | https://catalog.data.gov/dataset/fsis-foodkeeper-data |
| File used | https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls |

In [1]:
# Setup
import io, os, urllib.request
import pandas as pd
import numpy as np

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

FOODKEEPER_URL = "https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls"
LOCAL_FILE = "FoodKeeper-Data.xls"
print("pandas", pd.__version__)

pandas 3.0.5


## 2. Load the dataset

The cell below downloads the file straight from the USDA server so this notebook is
reproducible — anyone running it gets the data from the source, not from a copy we made.

If the download is blocked (some networks block the USDA host), the cell falls back to a local
copy of `FoodKeeper-Data.xls` placed next to the notebook. In Google Colab you can upload it
with the file browser on the left.

In [2]:
def load_foodkeeper():
    if os.path.exists(LOCAL_FILE):
        print("Using local file:", LOCAL_FILE)
        return pd.ExcelFile(LOCAL_FILE)
    try:
        print("Downloading from USDA ...")
        raw = urllib.request.urlopen(FOODKEEPER_URL, timeout=60).read()
        with open(LOCAL_FILE, "wb") as f:
            f.write(raw)
        print("Downloaded", len(raw), "bytes")
        return pd.ExcelFile(LOCAL_FILE)
    except Exception as e:
        raise SystemExit(
            "Could not download the FoodKeeper file (%s).\n"
            "Download it manually from %s and place it next to this notebook as %s."
            % (e, FOODKEEPER_URL, LOCAL_FILE))

# .xls files need the xlrd engine
try:
    import xlrd
except ImportError:
    print("installing xlrd ...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlrd"], check=True)

book = load_foodkeeper()
print("Sheets:", book.sheet_names)

Using local file: FoodKeeper-Data.xls
Sheets: ['Version', 'Category', 'Product', 'CookingTips', 'CookingMethods', 'Data Dictionary']


In [3]:
products   = book.parse("Product")
categories = book.parse("Category")

products["Name"] = products["Name"].astype(str)

print("Total foods in FoodKeeper:", len(products))
categories

Total foods in FoodKeeper: 661


,ID,Category_Name,Subcategory_Name
0,1,Baby Food,NaN
1,2,Baked Goods,Bakery
2,3,Baked Goods,Baking and Cooking
3,4,Baked Goods,Refrigerated Dough
4,5,Beverages,NaN
5,6,"Condiments, Sauces & Canned Goods",NaN
6,7,Dairy Products & Eggs,NaN
7,8,Food Purchased Frozen,NaN
8,9,"Grains, Beans & Pasta",NaN
9,10,Meat,Fresh


## 3. One detail that matters for the code

FoodKeeper does not store a single "fridge days" number. It stores several, and they mean
different things:

| Column group | What the clock starts from |
|---|---|
| `DOP_Refrigerate_*` | the **date of purchase** |
| `Refrigerate_*` | simply refrigerated |
| `Refrigerate_After_Opening_*` | the day the package was **opened** |

This is not a formatting detail. Carrots last 2–3 weeks **from purchase**. Hard cheese lasts
3–4 weeks **from opening** — an unopened block keeps for months. If the app treats both as
"days since added", unopened cheese will be flagged as expiring when it is not.

Units also vary per row (`Days`, `Weeks`, `Months`), so everything is normalised to days.

In [4]:
TO_DAYS = {"Days": 1, "Weeks": 7, "Months": 30, "Years": 365}

def to_days(value, metric):
    """Convert a FoodKeeper value + unit into days. Returns NaN if either is missing."""
    if pd.isna(value) or pd.isna(metric):
        return np.nan
    return float(value) * TO_DAYS.get(str(metric).strip(), np.nan)

def unopened_window(row):
    """Fridge life before the package is opened, with the basis it is measured from."""
    for lo, hi, met, basis in [
        ("DOP_Refrigerate_Min", "DOP_Refrigerate_Max", "DOP_Refrigerate_Metric", "from purchase"),
        ("Refrigerate_Min",     "Refrigerate_Max",     "Refrigerate_Metric",     "refrigerated"),
    ]:
        if not pd.isna(row.get(lo)):
            return to_days(row[lo], row[met]), to_days(row[hi], row[met]), basis, ""
    # No number. FoodKeeper sometimes puts a text instruction in the metric column instead.
    for met in ["DOP_Refrigerate_Metric", "Refrigerate_Metric"]:
        text = row.get(met)
        if isinstance(text, str) and text.strip():
            return np.nan, np.nan, "no number given", text.strip()
    return np.nan, np.nan, "not listed", ""

def opened_window(row):
    """Fridge life counted from the day the package was opened."""
    lo, hi, met = ("Refrigerate_After_Opening_Min",
                   "Refrigerate_After_Opening_Max",
                   "Refrigerate_After_Opening_Metric")
    if pd.isna(row.get(lo)):
        return np.nan, np.nan
    return to_days(row[lo], row[met]), to_days(row[hi], row[met])

print("helpers defined")

helpers defined


## 4. Pull the demo foods

Samira asked for at least one food from each food group rather than a long list, so the demo
proves the app works across categories instead of on one lucky item. These are the foods
selected, with the FoodKeeper row each one matches.

In [5]:
category_label = dict(zip(
    categories["ID"],
    categories["Category_Name"].astype(str)
    + categories["Subcategory_Name"].fillna("").apply(lambda s: " - " + s if s else "")
))

# (FoodKeeper Name, text that must appear in Name_subtitle or None)
TARGETS = [
    ("Milk",                               "plain"),      # dairy
    ("Yogurt",                             None),         # dairy
    ("Cheese",                             "hard"),       # dairy
    ("Eggs",                               "in shell"),   # eggs
    ("Lettuce",                            "leaf"),       # vegetable
    ("Carrots, parsnips",                  None),         # vegetable
    ("Broccoli and broccoli raab (rapini)", None),        # vegetable
    ("Apples",                             None),         # fruit
    ("Bananas",                            None),         # fruit
    ("Chicken parts",                      "boneless"),   # poultry
    ("Tofu",                               None),         # vegetarian protein
    ("Cooked rice",                        None),         # prepared / leftovers
    ("Tomato sauce",                       "spaghetti"),  # packaged
]

def find_row(name, subtitle=None):
    match = products[products["Name"].str.lower() == name.lower()]
    if subtitle is not None:
        match = match[match["Name_subtitle"].fillna("").str.lower().str.contains(subtitle.lower())]
    if len(match) == 0:
        raise LookupError("No FoodKeeper row for %r / %r" % (name, subtitle))
    return match.iloc[0]

records = []
for name, subtitle in TARGETS:
    row = find_row(name, subtitle)
    lo, hi, basis, note = unopened_window(row)
    o_lo, o_hi = opened_window(row)
    records.append({
        "food":            name + (" (%s)" % subtitle if subtitle else ""),
        "category":        category_label.get(row["Category_ID"], "?"),
        "unopened_min":    lo,
        "unopened_max":    hi,
        "unopened_basis":  basis,
        "opened_min":      o_lo,
        "opened_max":      o_hi,
        "note":            note,
    })

shelf_life = pd.DataFrame(records)
shelf_life

,food,category,unopened_min,unopened_max,unopened_basis,opened_min,opened_max,note
0,Milk (plain),Dairy Products & Eggs,NaN,NaN,no number given,NaN,NaN,Package use-by date
1,Yogurt,Dairy Products & Eggs,7.0,14.0,from purchase,NaN,NaN,
2,Cheese (hard),Dairy Products & Eggs,180.0,180.0,from purchase,21.0,28.0,
3,Eggs (in shell),Dairy Products & Eggs,21.0,35.0,from purchase,NaN,NaN,
4,Lettuce (leaf),Produce - Fresh Vegetables,3.0,7.0,from purchase,NaN,NaN,
5,"Carrots, parsnips",Produce - Fresh Vegetables,14.0,21.0,from purchase,NaN,NaN,
6,Broccoli and broccoli raab (rapini),Produce - Fresh Vegetables,3.0,5.0,from purchase,NaN,NaN,
7,Apples,Produce - Fresh Fruits,28.0,42.0,from purchase,NaN,NaN,
8,Bananas,Produce - Fresh Fruits,3.0,3.0,refrigerated,NaN,NaN,
9,Chicken parts (boneless),Poultry - Fresh,1.0,2.0,from purchase,NaN,NaN,


## 5. Finding 1 — milk has no number

FoodKeeper gives no day count for milk. It says to follow the package use-by date.

This matters for the demo. Milk is the obvious food to show off a fridge app with, and it is
the one food here our lookup cannot answer. Two options: have the app return `UNKNOWN` for
milk, or read the printed date off the carton. Either way the demo should use **yogurt** as its
dairy example, which does have a number.

In [6]:
no_number = shelf_life[shelf_life["unopened_basis"] == "no number given"]
no_number[["food", "category", "unopened_basis", "note"]]

,food,category,unopened_basis,note
0,Milk (plain),Dairy Products & Eggs,no number given,Package use-by date


## 6. Finding 2 — the clock does not always start in the same place

Below, every food that has an "after opening" window. For hard cheese the two numbers are far
apart: months unopened, a few weeks once opened. Any food in this list needs the app to know
whether the item has been opened, not just when it was added.

In [7]:
both = shelf_life[shelf_life["opened_min"].notna()][
    ["food", "unopened_min", "unopened_max", "unopened_basis", "opened_min", "opened_max"]
]
both

,food,unopened_min,unopened_max,unopened_basis,opened_min,opened_max
2,Cheese (hard),180.0,180.0,from purchase,21.0,28.0
12,Tomato sauce (spaghetti),NaN,NaN,not listed,3.0,5.0


## 7. Export for Role 3

Written as a CSV so the shelf-life module can load it directly. The source URL is carried in
every row — if anyone questions a number later, the row says where it came from.

In [8]:
SOURCE = ("USDA FSIS FoodKeeper Data (EN), "
          "https://www.fsis.usda.gov/shared/data/EN/FoodKeeper-Data.xls")

export = shelf_life.copy()
export["source"] = SOURCE
export.to_csv("foodkeeper_shelf_life.csv", index=False)

print("Wrote foodkeeper_shelf_life.csv with %d rows" % len(export))
export.head(3)

Wrote foodkeeper_shelf_life.csv with 13 rows


,food,category,unopened_min,unopened_max,unopened_basis,opened_min,opened_max,note,source
0,Milk (plain),Dairy Products & Eggs,NaN,NaN,no number given,NaN,NaN,Package use-by date,"USDA FSIS FoodKeeper Data (EN), https://www.fs..."
1,Yogurt,Dairy Products & Eggs,7.0,14.0,from purchase,NaN,NaN,,"USDA FSIS FoodKeeper Data (EN), https://www.fs..."
2,Cheese (hard),Dairy Products & Eggs,180.0,180.0,from purchase,21.0,28.0,,"USDA FSIS FoodKeeper Data (EN), https://www.fs..."


## 8. Summary

- The FoodSafety.gov chart our shelf-life data was built from does not cover produce or dairy.
  Neither does the FDA chart. Both point to USDA FoodKeeper instead.
- FoodKeeper supplied values for every food group the demo needs, pulled here directly from
  the USDA file so the numbers are reproducible rather than retyped.
- **Milk has no day count** in FoodKeeper. It must return `UNKNOWN` or fall back to the
  printed date. The demo should use yogurt for dairy.
- **The countdown starts at different points** depending on the food — purchase date for most,
  opening date for others. The shelf-life module needs to handle both, otherwise unopened
  items get flagged early.

### Sources

- USDA FSIS FoodKeeper Data — https://catalog.data.gov/dataset/fsis-foodkeeper-data
- FoodSafety.gov Cold Food Storage Charts — https://www.foodsafety.gov/food-safety-charts/cold-food-storage-charts
- FDA Refrigerator & Freezer Storage Chart — https://www.fda.gov/media/74435/download

No API keys or credentials are used in this notebook.